# Final Project Notebook

> This notebook is packaged for direct use after installing dependencies from `requirements.txt`. It is designed to run in VS Code Jupyter, or in an already installed Jupyter Notebook environment, on Windows, macOS, and Linux.

## Before running
1. Create and activate a Python virtual environment.
2. Run `pip install -r requirements.txt` in this folder.
3. Open this notebook and run the cells from top to bottom.
4. If you want to use another input image, replace the image path in the Stage 1 image-loading cell.

## First-run note
The first execution needs an internet connection because Hugging Face and Segment Anything model files are downloaded automatically.

In [ ]:
# Verify that the Python environment is ready before running the project
from importlib.util import find_spec
import platform
import sys

REQUIRED_PACKAGES = {
    "cv2": "opencv-python",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "torch": "torch",
    "PIL": "Pillow",
    "transformers": "transformers",
    "ipywidgets": "ipywidgets",
    "segment_anything": "segment-anything",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if find_spec(module_name) is None
]

print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.system()} {platform.release()}")

if missing_packages:
    missing_text = ", ".join(sorted(set(missing_packages)))
    raise ModuleNotFoundError(
        "Missing required packages: "
        f"{missing_text}. Install them with `pip install -r requirements.txt` in this folder, "
        "then rerun this cell."
    )

print("Environment check passed.")

In [ ]:
# Import required libraries and resolve project paths in a cross-platform way
from pathlib import Path
import urllib.request

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from transformers import pipeline
from segment_anything import sam_model_registry, SamPredictor

def find_project_root():
    search_roots = [Path.cwd(), *Path.cwd().parents]
    for base_path in search_roots:
        if (base_path / "Final_Project.ipynb").exists():
            return base_path
        nested_path = base_path / "personal" / "fur_rendering_project"
        if (nested_path / "Final_Project.ipynb").exists():
            return nested_path
    raise FileNotFoundError(
        "Could not locate the project folder automatically. Open the notebook from the extracted project folder, "
        "or set PROJECT_ROOT manually in this cell."
    )

def iter_candidate_images(project_root):
    candidate_roots = [project_root, project_root / "image_folder"]
    for root in candidate_roots:
        if not root.exists():
            continue
        for pattern in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
            for candidate in sorted(root.glob(pattern)):
                if candidate.is_file() and not candidate.name.startswith("._"):
                    yield candidate

def resolve_default_image_path(project_root):
    preferred_candidates = [
        project_root / "20260517-141241.jpg",
        project_root / "image_folder" / "20260517-141241.jpg",
        project_root / "image_folder" / "mug1.jpg",
    ]
    for candidate in preferred_candidates:
        if candidate.exists():
            return candidate

    for candidate in iter_candidate_images(project_root):
        return candidate

    raise FileNotFoundError(
        "No input image was found in the project folder or image_folder. Add a .jpg or .png file next to this notebook, "
        "or place images under image_folder, or set DEFAULT_IMAGE_PATH manually in this cell."
    )

PROJECT_ROOT = find_project_root()
DEFAULT_IMAGE_PATH = resolve_default_image_path(PROJECT_ROOT)
OUTPUT_DIR = PROJECT_ROOT / "notebook_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def show_image_file(image_path, title, cmap=None, figsize=(5, 5)):
    loaded_image = Image.open(image_path)
    image_array = np.array(loaded_image)
    plt.figure(figsize=figsize)
    if image_array.ndim == 2 or cmap is not None:
        plt.imshow(image_array, cmap=cmap or "gray")
    else:
        plt.imshow(image_array)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

print(f"Project root: {PROJECT_ROOT}")
print(f"Default image path: {DEFAULT_IMAGE_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

# **Stage 1: Approximate 2.5D Surface / Depth Map**

In [ ]:
# 1. Load depth estimation model
device = 0 if torch.cuda.is_available() else -1

depth_pipe = pipeline(
    task="depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
    device=device
)

print("Depth estimation model loaded.")

In [ ]:
# 2. Load the input image from the project folder
# If you want to use another image, replace DEFAULT_IMAGE_PATH with PROJECT_ROOT / "your-image.jpg".
image_path = DEFAULT_IMAGE_PATH
if not image_path.exists():
    raise FileNotFoundError(f"Input image not found: {image_path}")

image = Image.open(image_path).convert("RGB")
image_np = np.array(image)
H, W = image_np.shape[:2]

print("Loaded image:", image_path)
print(f"Image size: {W} x {H}")

In [ ]:
# 3. Generate depth map
result = depth_pipe(image)

depth = np.array(result["depth"]).astype(np.float32)

depth = cv2.resize(depth, image.size)

depth_norm = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

depth_gray = (depth_norm * 255).astype(np.uint8)

depth_color = cv2.applyColorMap(depth_gray, cv2.COLORMAP_INFERNO)

depth_color = cv2.cvtColor(depth_color, cv2.COLOR_BGR2RGB)

print("Depth map generated.")

In [ ]:
# 4. Display original image and depth map
image_np = np.array(image)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_np)
plt.axis("off")
plt.title("Input Mug Image")

plt.subplot(1, 2, 2)
plt.imshow(depth_color)
plt.axis("off")
plt.title("Estimated Depth Map")

plt.tight_layout()
plt.show()

In [ ]:
# 5. Save depth map result
depth_output_path = OUTPUT_DIR / "estimated_depth_map.png"
comparison_output_path = OUTPUT_DIR / "depth_estimation_result.png"

cv2.imwrite(
    str(depth_output_path),
    cv2.cvtColor(depth_color, cv2.COLOR_RGB2BGR)
 )

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_np)
plt.axis("off")
plt.title("Input Mug Image")

plt.subplot(1, 2, 2)
plt.imshow(depth_color)
plt.axis("off")
plt.title("Estimated Depth Map")

plt.tight_layout()
plt.savefig(comparison_output_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved depth map:", depth_output_path)
print("Saved comparison image:", comparison_output_path)

In [ ]:
# 6. Local notebook mode: files are already saved under OUTPUT_DIR
print("Depth outputs are available at:")
print(" -", depth_output_path)
print(" -", comparison_output_path)

# **Stage 2: Mug Matte Generation**

In [ ]:
# 1. Load pretrained SAM model
sam_checkpoint = OUTPUT_DIR / "sam_vit_b_01ec64.pth"
if not sam_checkpoint.exists():
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
        sam_checkpoint,
    )

model_type = "vit_b"
sam = sam_model_registry[model_type](checkpoint=str(sam_checkpoint))
sam.to(device="cuda" if torch.cuda.is_available() else "cpu")
predictor = SamPredictor(sam)

print("SAM checkpoint ready:", sam_checkpoint)

In [ ]:
# 2. Set the Stage 1 input image for SAM

predictor.set_image(image_np)

height, width = image_np.shape[:2]

center_x = width // 2
center_y = height // 2

print("Image set for SAM.")
print("Image size:", width, "x", height)
print("Image center:", center_x, center_y)

In [ ]:
# 3. Generate mug matte using the image center as the foreground prompt

input_point = np.array([[center_x, center_y]])
input_label = np.array([1])

masks, scores, logits = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True
)

best_mask = masks[np.argmax(scores)]
best_score = np.max(scores)

print("Mug matte generated.")
print("Best SAM score:", round(float(best_score), 3))

In [ ]:
# 4. Validate matte

mask_area = np.sum(best_mask)
image_area = height * width
mask_ratio = mask_area / image_area

mug_matte = best_mask.astype(np.uint8)

if mask_ratio < 0.03:
    raise ValueError("The detected mug matte is too small. Please use an image with one clear mug near the center.")

if mask_ratio > 0.85:
    raise ValueError("The detected mug matte is too large (>85% of image). SAM may have selected the entire image.")
elif mask_ratio < 0.02:
    raise ValueError("The detected mug matte is too small (<2% of image). SAM may have failed.")

# 检查mask是否触碰太多边界（可能是背景）


border_pixels = np.sum(mug_matte[0, :]) + np.sum(mug_matte[-1, :]) + \
                np.sum(mug_matte[:, 0]) + np.sum(mug_matte[:, -1])
border_ratio = border_pixels / (2 * (height + width))

if border_ratio > 0.5:
    print(f"Warning: Mask touches {border_ratio*100:.1f}% of image borders. This might include background.")

if best_mask[center_y, center_x] == False:
    raise ValueError("The image center is not inside the mug matte. Please use a centered mug image.")

top_touch = np.any(best_mask[0:5, :])
bottom_touch = np.any(best_mask[-5:, :])
left_touch = np.any(best_mask[:, 0:5])
right_touch = np.any(best_mask[:, -5:])

border_touch_count = sum([top_touch, bottom_touch, left_touch, right_touch])

if border_touch_count >= 2:
    raise ValueError("The mug matte touches too many image borders. Please use a centered mug image with clean background.")

print("Mug matte passed validation.")
print("Mask area ratio:", round(float(mask_ratio), 3))
print("Border touch count:", border_touch_count)

In [ ]:
# 5. Display the mug matte and overlay result

mug_matte = best_mask.astype(np.uint8)

mug_matte_255 = mug_matte * 255

matte_overlay = image_np.copy()
matte_overlay[mug_matte == 1] = (
    matte_overlay[mug_matte == 1] * 0.5 +
    np.array([255, 0, 0]) * 0.5
)

matte_overlay = matte_overlay.astype(np.uint8)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image_np)
plt.axis("off")
plt.title("Input Mug Image")

plt.subplot(1, 3, 2)
plt.imshow(mug_matte_255, cmap="gray")
plt.axis("off")
plt.title("Mug Matte")

plt.subplot(1, 3, 3)
plt.imshow(matte_overlay)
plt.axis("off")
plt.title("Mug Matte Overlay")

plt.tight_layout()
plt.show()

In [ ]:
# 6. Save mug matte and overlay image
mug_matte = best_mask.astype(np.uint8)
mug_matte_255 = mug_matte * 255

matte_overlay = image_np.copy()
matte_overlay[mug_matte == 1] = (
    matte_overlay[mug_matte == 1] * 0.5 +
    np.array([255, 0, 0]) * 0.5
)
matte_overlay = matte_overlay.astype(np.uint8)

matte_output_path = OUTPUT_DIR / "mug_matte.png"
matte_overlay_output_path = OUTPUT_DIR / "mug_matte_overlay.png"

cv2.imwrite(str(matte_output_path), mug_matte_255)

cv2.imwrite(
    str(matte_overlay_output_path),
    cv2.cvtColor(matte_overlay, cv2.COLOR_RGB2BGR)
 )

print("Saved mug matte:", matte_output_path)
print("Saved mug matte overlay:", matte_overlay_output_path)

In [ ]:
# Local notebook mode: files are already saved under OUTPUT_DIR
print("Mug matte outputs are available at:")
print(" -", matte_output_path)
print(" -", matte_overlay_output_path)

# **Stage 3: Approximate Surface / Normal Guidance**

In [ ]:
# 1. Create vertices from depth map
vertices = []
vertex_map = np.full((height, width), -1, dtype=int)

for i in range(height):
    for j in range(width):

        if mug_matte[i, j] > 0:
            vertex_map[i, j] = len(vertices)
            x, y = j, i
            z = depth_norm[i, j] * min(height, width) * 1.0
            vertices.append([x, y, z])

vertices = np.array(vertices)
print(f"✓ Created {len(vertices)} vertices from depth map")

In [ ]:
# 2. Convert 2x2 blocks into triangles (following Pouli et al. 2010)
triangles = []

for i in range(height - 1):
    for j in range(width - 1):
        v = [vertex_map[i, j], vertex_map[i, j+1],
             vertex_map[i+1, j], vertex_map[i+1, j+1]]

        if all(x >= 0 for x in v):
            # Each 2x2 block creates 2 triangles
            triangles.append([v[0], v[1], v[2]])
            triangles.append([v[1], v[3], v[2]])

triangles = np.array(triangles)
print(f"✓ Created {len(triangles)} triangles")

In [ ]:
# 3. Compute surface normals for each vertex
vertex_normals = np.zeros((len(vertices), 3))

for tri in triangles:
    v0, v1, v2 = vertices[tri]

    # Compute face normal using cross product
    edge1 = v1 - v0
    edge2 = v2 - v0
    face_normal = np.cross(edge1, edge2)

    # Accumulate face normal to all 3 vertices
    vertex_normals[tri[0]] += face_normal
    vertex_normals[tri[1]] += face_normal
    vertex_normals[tri[2]] += face_normal

# Normalize vertex normals
norms = np.linalg.norm(vertex_normals, axis=1, keepdims=True)
vertex_normals = vertex_normals / (norms + 1e-8)

print("✓ Computed vertex normals")

In [ ]:
# 4. Render vertex normals to image for visualization
normal_image = np.zeros((height, width, 3))

for i in range(height):
    for j in range(width):
        v_idx = vertex_map[i, j]
        if v_idx >= 0:
            # Map normal from [-1,1] to [0,1] for RGB visualization
            normal_image[i, j] = (vertex_normals[v_idx] + 1.0) * 0.5

# Apply mask
normal_image[~mug_matte.astype(bool)] = 0

# Convert to uint8
normal_vis = (normal_image * 255).astype(np.uint8)

print("✓ Generated normal map visualization")

In [ ]:
# 5. Display results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(depth_color)
plt.axis("off")
plt.title("Depth Map", fontsize=12, fontweight='bold')

plt.subplot(1, 2, 2)
plt.imshow(normal_vis)
plt.axis("off")
plt.title("Surface Normals", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 6. Save normal map
normal_output = OUTPUT_DIR / "stage3_surface_normals.png"
cv2.imwrite(str(normal_output), cv2.cvtColor(normal_vis, cv2.COLOR_RGB2BGR))
print(f"Saved: {normal_output}")
print("Stage 3 complete: Surface normals generated")

# **Stage 4: Fur Generation (Paper Basis: Section 2.2, Eq. 1 and Eq. 2)**

Use the Stage 3 mesh and normals directly. Hair roots are sampled on mesh triangles with barycentric coordinates, and each strand is grown by particle-style Euler integration.

In [ ]:
# 1. Generate fur strands from the existing mesh using barycentric root placement
image_float = image_np.astype(np.float32) / 255.0
mask = mug_matte.astype(np.uint8)
depth_scale = max(float(vertices[:, 2].max()), 1.0)

FUR_DENSITY = 0.35  # 🆕 增加密度（从0.18提升到0.35）
FUR_LENGTH_MIN = 6.0
FUR_LENGTH_MAX = 15.0
GRAVITY_DIR = np.array([0.0, 1.0, 0.0], dtype=np.float32)
GRAVITY_STRENGTH = 0.18
DIRECTION_RANDOMNESS = 0.25
N_STEPS = 12
dt = 1.0 / N_STEPS

np.random.seed(42)
root_positions = []
root_normals = []
root_lengths = []

# 计算深度范围和中心点（用于后续自适应处理）
z_min = vertices[:, 2].min()
z_max = vertices[:, 2].max()
z_range = z_max - z_min + 1e-6
center = vertices.mean(axis=0)

for tri in triangles:
    v0, v1, v2 = vertices[tri].astype(np.float32)
    area = 0.5 * np.linalg.norm(np.cross(v1 - v0, v2 - v0))

    # 边缘增强密度
    z_values = [v0[2], v1[2], v2[2]]
    z_range_tri = max(z_values) - min(z_values)
    edge_boost = 1.0 + min(z_range_tri / (depth_scale + 1e-6) * 3.0, 2.5)

    expected_hairs = FUR_DENSITY * area * edge_boost
    n_roots = int(expected_hairs)
    if np.random.rand() < (expected_hairs - n_roots):
        n_roots += 1
    if n_roots == 0:
        continue

    n0, n1, n2 = vertex_normals[tri[0]], vertex_normals[tri[1]], vertex_normals[tri[2]]
    for _ in range(n_roots):
        u = np.random.rand()
        v = np.random.rand()
        if u + v > 1.0:
            u = 1.0 - u
            v = 1.0 - v
        w = 1.0 - u - v
        root = w * v0 + u * v1 + v * v2
        root_normal = w * n0 + u * n1 + v * n2
        root_normal = root_normal / (np.linalg.norm(root_normal) + 1e-8)

        # 增加长度随机性
        length_variation = np.random.uniform(0.6, 1.4)  # ±40%变化
        base_length = np.random.uniform(FUR_LENGTH_MIN, FUR_LENGTH_MAX)
        length = base_length * length_variation * (0.7 + 0.3 * np.clip(root[2] / depth_scale, 0.0, 1.0))

        root_positions.append(root)
        root_normals.append(root_normal)
        root_lengths.append(length)

root_positions = np.array(root_positions, dtype=np.float32)
root_normals = np.array(root_normals, dtype=np.float32)
root_lengths = np.array(root_lengths, dtype=np.float32)

# 生成毛发曲线
hairs = []
for root, normal, length in zip(root_positions, root_normals, root_lengths):
    # 原有的随机噪声
    direction_noise = np.random.uniform(-DIRECTION_RANDOMNESS, DIRECTION_RANDOMNESS, 3).astype(np.float32)

    # 径向方向（从中心向外）
    radial_dir = root - center
    radial_dir = radial_dir / (np.linalg.norm(radial_dir) + 1e-8)

    # 混合法线、径向、随机
    radial_strength = 0.35
    init_dir = (1.0 - radial_strength) * normal + radial_strength * radial_dir + direction_noise
    init_dir = init_dir / (np.linalg.norm(init_dir) + 1e-8)

    velocity = init_dir * length

    # 根据深度位置自适应调整重力
    normalized_depth = (root[2] - z_min) / z_range

    if normalized_depth > 0.6:  # 底部40%区域
        gravity_factor = 0.12  # 只保留12%重力
    elif normalized_depth > 0.3:  # 中部
        gravity_factor = 0.5
    else:  # 顶部
        gravity_factor = 1.0

    adaptive_gravity = GRAVITY_STRENGTH * gravity_factor

    # 粒子模拟
    position = root.copy()
    points = [position.copy()]
    for _ in range(N_STEPS):
        velocity = velocity + GRAVITY_DIR * adaptive_gravity * length * dt
        position = position + velocity * dt
        points.append(position.copy())

    hairs.append(np.array(points, dtype=np.float32))

print(f"Mesh vertices: {len(vertices)}")
print(f"Mesh triangles: {len(triangles)}")
print(f"Generated {len(hairs)} hair strands via barycentric root placement")

# **Stage 5: Rendering with Kajiya-Kay Model (Paper Basis: Section 2.4)**

Render the generated strands as tapered ribbon approximations, then shade them with a Kajiya-Kay style highlight model before compositing.

In [ ]:
# 1. Render fur strands with Kajiya-Kay style shading
luminance = 0.2126 * image_float[:, :, 0] + 0.7152 * image_float[:, :, 1] + 0.0722 * image_float[:, :, 2]
masked_luminance = luminance[mask > 0]
threshold = np.percentile(masked_luminance, 85) if masked_luminance.size else 0.8
bright_mask = (luminance > threshold) & (mask > 0)

if bright_mask.sum() > 0:
    by, bx = np.where(bright_mask)
    cy, cx = by.mean(), bx.mean()
    light_dir = np.array([(cx / W - 0.5), (cy / H - 0.5), 0.8], dtype=np.float32)
else:
    light_dir = np.array([0.25, -0.15, 0.95], dtype=np.float32)

light_dir = light_dir / np.linalg.norm(light_dir)
light_color = np.array([1.0, 0.95, 0.9], dtype=np.float32)
view_dir = np.array([0.0, 0.0, 1.0], dtype=np.float32)


# 在mask范围内对杯子本体做高斯模糊，只保留粗略颜色信息
# 这样毛发采样时不会继承原图的强反光、细节纹理和局部高光
blurred_image = cv2.GaussianBlur(image_float, (0, 0), sigmaX=18, sigmaY=18)

mask_3d = mask[:, :, np.newaxis].astype(np.float32)

# 只替换mask内部为模糊图，mask外仍然保持原图
color_sample_image = image_float.copy()
color_sample_image = (
    mask_3d * blurred_image +
    (1.0 - mask_3d) * image_float
)

# 从模糊后的mask区域采样平均颜色作为基础色
masked_pixels = color_sample_image[mask > 0]




if len(masked_pixels) > 0:
    # 使用中位数更稳健（避免异常值影响）
    base_color = np.median(masked_pixels, axis=0).astype(np.float32)
    # 稍微调暗作为根部颜色
    base_color = base_color * 0.85
    # 尖端颜色更亮
    tip_color = np.clip(base_color * 1.25, 0.0, 1.0)
else:
    # fallback到默认颜色
    base_color = np.array([0.68, 0.48, 0.40], dtype=np.float32)
    tip_color = np.array([0.86, 0.70, 0.62], dtype=np.float32)

print(f"Sampled base color: {base_color}")
print(f"Sampled tip color: {tip_color}")

fur_layer = np.zeros((H, W, 3), dtype=np.float32)
fur_alpha = np.zeros((H, W), dtype=np.float32)

for hair in hairs:
    # 获取毛发根部位置并采样颜色
    root_pos = hair[0]
    root_x = int(np.clip(root_pos[0], 0, W - 1))
    root_y = int(np.clip(root_pos[1], 0, H - 1))

    # 从原图该位置采样颜色
    if mask[root_y, root_x] > 0:
        sampled_color = color_sample_image[root_y, root_x]
        hair_base_color = sampled_color * 0.85  # 根部稍微调暗
        hair_tip_color = np.clip(sampled_color * 1.2, 0.0, 1.0)  # 尖端更亮
    else:
        # fallback到全局颜色
        hair_base_color = base_color
        hair_tip_color = tip_color

    n_points = len(hair)
    color_offset = np.random.uniform(-0.04, 0.04, 3).astype(np.float32)

    for j in range(n_points - 1):
        p0, p1 = hair[j], hair[j + 1]
        tangent = p1 - p0
        tangent_norm = np.linalg.norm(tangent)
        if tangent_norm < 1e-8:
            continue
        tangent = tangent / tangent_norm
        TdotL = np.dot(tangent, light_dir)
        diffuse = np.sqrt(max(1.0 - TdotL * TdotL, 0.0))

        t = j / max(n_points - 1, 1)
        # 使用每根毛发独立的颜色
        strand_color = np.clip((1.0 - t) * hair_base_color + t * hair_tip_color + color_offset, 0.0, 1.0)
        shaded_color = np.clip(strand_color * light_color * (0.5 + 0.5 * diffuse), 0.0, 1.0)

        x0, y0 = int(round(p0[0])), int(round(p0[1]))
        x1, y1 = int(round(p1[0])), int(round(p1[1]))
        W_px = max(2, int(4.8 * (1.0 - 0.35 * t)))
        alpha = 1

        cv2.line(fur_layer, (x0, y0), (x1, y1), shaded_color.tolist(), thickness=W_px, lineType=cv2.LINE_AA)
        cv2.line(fur_alpha, (x0, y0), (x1, y1), float(alpha), thickness=W_px, lineType=cv2.LINE_AA)

# The center looks sparse because silhouette regions get many overlapping strand projections,
# while front-facing regions show isolated strands with less overlap. Add a light underfur fill
# into fur color, but keep a lighter composite alpha so the final result does not collapse into a blob.

# 只在mask内部模糊，避免边缘黑影
mask_3d = mask[:, :, np.newaxis].astype(np.float32)

# 轻微模糊毛层
fur_layer_blurred = cv2.GaussianBlur(fur_layer, (0, 0), sigmaX=1.2, sigmaY=1.2)
fur_layer = np.clip(0.85 * fur_layer + 0.15 * fur_layer_blurred, 0.0, 1.0)

# Alpha通道轻微模糊（边缘羽化）
fur_alpha_blurred = cv2.GaussianBlur(fur_alpha, (0, 0), sigmaX=1.5, sigmaY=1.5)

# 🔑关键：只在mask内部应用模糊，边缘直接用原值
fur_alpha = fur_alpha_blurred * mask + fur_alpha * (1 - mask)

# 轻微增强alpha
fur_alpha = np.clip(fur_alpha * 1.05, 0.0, 0.92)


plt.figure(figsize=(6, 6))
plt.imshow(fur_layer)
plt.axis("off")
plt.title("Rendered Fur Layer")
plt.tight_layout()
plt.show()

print(f"Light direction: {light_dir}")
print("Rendering complete.")

# **Stage 6: Compositing & Final Result**

Blend the fur back onto the original image by first adding a diffuse underlayer and then alpha-compositing the rendered fur strands.

In [ ]:
# 1. Composite the fur layer and save the final result
result_float = image_float.copy()

# 在mask内部，用底色替换原图（去除杯子及其阴影）
diffuse_underlayer = np.zeros_like(result_float)
for c in range(3):
    diffuse_underlayer[:, :, c] = base_color[c]

# 完全替换mask内部为底色
mask_3d = mask[:, :, np.newaxis].astype(np.float32)
result_float = (1.0 - mask_3d) * result_float + mask_3d * diffuse_underlayer

# 然后轻微混合原图的光照信息（保留明暗变化但去除阴影）
distance_from_edge = cv2.distanceTransform(mask.astype(np.uint8), cv2.DIST_L2, 5)
distance_normalized = np.clip(distance_from_edge / 12.0, 0.0, 1.0)
blend_original = distance_normalized * 0.15  # 只保留15%的原图信息

for c in range(3):
    result_float[:, :, c] = (1.0 - blend_original) * result_float[:, :, c] + blend_original * image_float[:, :, c]

alpha_3d = np.clip(fur_alpha, 0.0, 1.0)[:, :, np.newaxis]
result_float = np.clip((1.0 - alpha_3d) * result_float + alpha_3d * fur_layer, 0.0, 1.0)



fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(fur_layer)
axes[0].axis("off")
axes[0].set_title("Fur Layer")
axes[1].imshow(result_float)
axes[1].axis("off")
axes[1].set_title("Final: Furry Mug")
plt.tight_layout()
plt.show()

final_output_path = OUTPUT_DIR / "final_project_furry_mug.png"
Image.fromarray((result_float * 255).astype(np.uint8)).save(final_output_path)
print(f"Saved {final_output_path}")
show_image_file(final_output_path, "Reloaded Saved Final Output")


# **Stage 7: Full Pipeline Visualization**

Summarize the final pipeline from original image to final furred result so the notebook presents all intermediate evidence in one figure.

In [ ]:
# 1. Visualize and save the full pipeline overview
pipeline_normals = np.zeros((H, W, 3), dtype=np.float32)
for y in range(H):
    for x in range(W):
        v_idx = vertex_map[y, x]
        if v_idx >= 0:
            pipeline_normals[y, x] = (vertex_normals[v_idx] + 1.0) * 0.5

pipeline_normals[mask == 0] = 0
depth_vis = cv2.applyColorMap((depth_norm * 255).astype(np.uint8), cv2.COLORMAP_INFERNO)
depth_vis = cv2.cvtColor(depth_vis, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(image_np)
axes[0, 0].axis("off")
axes[0, 0].set_title("1. Original")
axes[0, 1].imshow(mask, cmap="gray")
axes[0, 1].axis("off")
axes[0, 1].set_title("2. Mask")
axes[0, 2].imshow(depth_vis)
axes[0, 2].axis("off")
axes[0, 2].set_title("3. Depth")
axes[1, 0].imshow(pipeline_normals)
axes[1, 0].axis("off")
axes[1, 0].set_title("4. Normals")
axes[1, 1].imshow(fur_layer)
axes[1, 1].axis("off")
axes[1, 1].set_title("5. Fur Layer")
axes[1, 2].imshow(result_float)
axes[1, 2].axis("off")
axes[1, 2].set_title("6. Final")
plt.tight_layout()
pipeline_output_path = OUTPUT_DIR / "final_project_pipeline_overview.png"
plt.savefig(pipeline_output_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Pipeline complete: {pipeline_output_path}")
show_image_file(pipeline_output_path, "Reloaded Saved Pipeline Overview", figsize=(10, 7))